In [ ]:
"""
MooVision Prediction Distribution Analysis
------------------------------------------
Reads all per-video prediction JSONs from a directory and produces
Altair charts saved to an HTML file.

Usage:
    python visualize_predictions.py --input_dir /path/to/prediction/jsons
                                    --output    predictions_analysis.html
"""

import json
import argparse
from pathlib import Path

import pandas as pd
import altair as alt
import sys

from config import ROOT_DIR

BASELINE_MODEL_OUTPUT_DIR = ROOT_DIR / "data" / "results" / "metadata" / "baseline"
YOLO_MODEL_OUTPUT_DIR = ROOT_DIR / "results" / "metadata" / "random" / "yolo"
SEQ_MODEL_OUTPUT_DIR = ROOT_DIR / "results" / "metadata" / "random" / "seq-nms"

In [ ]:
def load_predictions(input_dir: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
        video_df  – one row per video
        event_df  – one row per event (exploded)
    """
    video_rows, event_rows = [], []

    for path in sorted(Path(input_dir).glob("*.json")):
        with open(path) as f:
            data = json.load(f)

        vid = data["identifier"]
        pen = _extract_pen(data.get("video_path", ""))
        stage = _extract_stage(data.get("video_path", ""))

        video_rows.append({
            "video":                   vid,
            "pen":                     pen,
            "weaning_stage":           stage,
            "total_duration_sec":      data.get("total_duration_sec"),
            "cross_sucking_detected":  data.get("cross_sucking_detected", False),
            "num_events":              data.get("num_events", 0),
        })

        for i, ev in enumerate(data.get("events", [])):
            event_rows.append({
                "video":           vid,
                "pen":             pen,
                "weaning_stage":   stage,
                "event_idx":       i,
                "start_sec":       ev["start_sec"],
                "end_sec":         ev["end_sec"],
                "duration_sec":    ev["duration_sec"],
                "avg_confidence":  ev["avg_confidence"],
            })

    video_df = pd.DataFrame(video_rows)
    event_df = pd.DataFrame(event_rows) if event_rows else pd.DataFrame(
        columns=["video", "pen", "weaning_stage", "event_idx",
                 "start_sec", "end_sec", "duration_sec", "avg_confidence"]
    )
    return video_df, event_df


def _extract_pen(video_path: str) -> str:
    """Best-effort pen extraction from path string."""
    import re
    m = re.search(r"Pen\s*(\d+)", video_path, re.IGNORECASE)
    return f"Pen {m.group(1)}" if m else "Unknown"


def _extract_stage(video_path: str) -> str:
    for stage in ("PREWEANING", "WEANING", "POSTWEANING"):
        if stage.lower() in video_path.lower():
            return stage
    return "Unknown"

video_df, event_df = load_predictions(SEQ_MODEL_OUTPUT_DIR)

In [ ]:
event_df

In [ ]:
def chart_confidence_distribution(event_df: pd.DataFrame) -> alt.Chart:
    """Histogram of avg_confidence across all events."""
    return (
        alt.Chart(event_df, title="Confidence Score Distribution (all events)")
        .mark_bar(opacity=0.8, color="steelblue")
        .encode(
            alt.X("avg_confidence:Q",
                  bin=alt.Bin(step=0.02),
                  title="Avg Confidence"),
            alt.Y("count():Q", title="# Events"),
            tooltip=["count():Q"],
        )
        .properties(width=500, height=280)
    )

chart_confidence_distribution(event_df)

In [ ]:
def chart_confidence_by_pen(event_df: pd.DataFrame) -> alt.Chart:
    """Overlapping confidence histograms faceted by pen."""
    return (
        alt.Chart(event_df, title="Confidence Distribution by Pen")
        .mark_bar(opacity=0.8,color="steelblue")
        .encode(
            alt.X("avg_confidence:Q", bin=alt.Bin(step=0.02), title="Avg Confidence"),
            alt.Y("count():Q", title="# Events"),
            alt.Color("pen:N", title="Pen"),
            tooltip=["pen:N", "count():Q"],
        )
        .properties(width=500, height=280)
    )
chart_confidence_by_pen(event_df)

In [ ]:
def chart_events_per_video(video_df: pd.DataFrame) -> alt.Chart:
    """Bar chart: number of predicted events per video, coloured by pen."""
    df = video_df.sort_values("num_events", ascending=False).copy()
    df["short_name"] = df["video"].str.replace(r"\.mp4$", "", regex=True)
 
    return (
        alt.Chart(df, title="Predicted Events per Video")
        .mark_bar()
        .encode(
            alt.X("short_name:N",
                  sort="-y",
                  title="Video",
                  axis=alt.Axis(labelAngle=-45, labelLimit=200)),
            alt.Y("num_events:Q", title="# Predicted Events"),
            alt.Color("pen:N", title="Pen"),
            tooltip=["short_name:N", "pen:N", "weaning_stage:N", "num_events:Q"],
        )
        .properties(width=700, height=300)
    )
chart_events_per_video(video_df)

In [ ]:
def chart_event_duration_distribution(event_df: pd.DataFrame) -> alt.Chart:
    """Histogram of event durations in seconds."""
    return (
        alt.Chart(event_df, title="Event Duration Distribution")
        .mark_bar(opacity=0.8, color="#F58518")
        .encode(
            alt.X("duration_sec:Q",
                  bin=alt.Bin(maxbins=30),
                  title="Duration (s)"),
            alt.Y("count():Q", title="# Events"),
            tooltip=["count():Q"],
        )
        .properties(width=500, height=280)
    )
chart_event_duration_distribution(event_df)

In [ ]:
def chart_temporal_density(event_df: pd.DataFrame) -> alt.Chart:
    """
    Scatter: event start time vs video (y-axis), sized by duration, coloured by confidence.
    Shows where within each video events are firing.
    """
    df = event_df.copy()
    df["short_name"] = df["video"].str.replace(r"\.mp4$", "", regex=True)
 
    return (
        alt.Chart(df, title="Temporal Density of Events Within Videos")
        .mark_point(filled=True, opacity=0.75)
        .encode(
            alt.X("start_sec:Q", title="Start Time (s)"),
            alt.Y("short_name:N", title="Video", sort="-x"),
            alt.Size("duration_sec:Q", title="Duration (s)",
                     scale=alt.Scale(range=[20, 300])),
            alt.Color("avg_confidence:Q",
                      scale=alt.Scale(scheme="viridis"),
                      title="Confidence"),
            tooltip=["short_name:N", "start_sec:Q", "end_sec:Q",
                     "duration_sec:Q", "avg_confidence:Q", "pen:N","weaning_stage:N"],
        )
        .properties(width=700, height=max(300, len(df["video"].unique()) * 22))
    )
chart_temporal_density(event_df)

In [ ]:
def chart_confidence_vs_duration(event_df: pd.DataFrame) -> alt.Chart:
    """Scatter: confidence vs duration, coloured by pen."""
    df = event_df.copy()
    df["short_name"] = df["video"].str.replace(r"\.mp4$", "", regex=True)
 
    return (
        alt.Chart(df, title="Confidence vs Event Duration")
        .mark_point(filled=True, opacity=0.7, size=60)
        .encode(
            alt.X("duration_sec:Q", title="Duration (s)"),
            alt.Y("avg_confidence:Q", title="Avg Confidence"),
            alt.Color("pen:N", title="Pen"),
            tooltip=["short_name:N", "pen:N", "weaning_stage:N",
                     "duration_sec:Q", "avg_confidence:Q"],
        )
        .properties(width=500, height=300)
    )
chart_confidence_vs_duration(event_df)